# Gerar Horarios com CSP

**Grupo:** IA25_P01_G02  
**Autores:** António Ferreira, Mafalda Barão, Gonçalo Gomes, Ruben Dias, João Morais

## Funcionalidades
1. Lê ficheiro ClassTT_01_tiny.txt (turmas, docentes, indisponibilidades)
2. Valida e faz diagnóstico (domínios, capacidade mínima)
3. Constrói um CSP (python-constraint) e procura solução viável
4. Mostra horários por turma e por docente
5. Exporta para CSV, PNG e PDF (cores por UC, hatch em aulas online)
6. Coloca banner docs/banner.png no topo dos PNG/PDF

## Importar Bibliotecas

In [ ]:
from collections import defaultdict
from dataclasses import dataclass
from typing import Dict, List, Tuple
import re, pathlib, sys, time, platform, os, colorsys, hashlib

from constraint import Problem, MinConflictsSolver  
import pandas as pd
import matplotlib.pyplot as plt

print("Bibliotecas importadas com sucesso!")

## Configuração Global

In [ ]:
# Ficheiro do dataset
DATA_PATH = "ClassTT_01_tiny.txt"

# Universo temporal: 5 dias * 4 blocos (2h cada) => 20 slots numerados 1..20
DAYS = ["Mon", "Tue", "Wed", "Thu", "Fri"]
BLOCKS_PER_DAY = 4
SLOTS = list(range(1, 5 * BLOCKS_PER_DAY + 1))  # [1..20]

# Tags de blocos e horários disponíveis
BLOCK_LABELS = ["B1", "B2", "B3", "B4"]
BLOCK_TO_TIME = {
    "B1": "09:00–11:00",
    "B2": "11:00–13:00",
    "B3": "14:00–16:00",
    "B4": "16:00–18:00",
}

print(f" Configuração: {len(DAYS)} dias × {BLOCKS_PER_DAY} blocos = {len(SLOTS)} slots")

## Funções Auxiliares

In [ ]:
def slot_day(slot: int) -> str:
    """Converte slot (1..20) no dia ('Mon'..'Fri')."""
    return DAYS[(slot - 1) // BLOCKS_PER_DAY]

def slot_to_day_and_block(slot: int) -> Tuple[str, str]:
    """Converte slot (1..20) em (dia, bloco: B1..B4)."""
    day_idx = (slot - 1) // BLOCKS_PER_DAY
    block_idx = (slot - 1) % BLOCKS_PER_DAY
    return DAYS[day_idx], BLOCK_LABELS[block_idx]

def read_section(raw: str, tag: str) -> List[str]:
    """Extrai seção do ficheiro baseada numa tag (#cc, #dsd, etc)."""
    pat = re.compile(rf"#{tag}[^\n]*\n(.*?)(?=\n#|$)", re.S)
    m = pat.search(raw)
    return [] if not m else [ln.strip() for ln in m.group(1).strip().splitlines() if ln.strip()]

# Testar conversão de slots
print("\n🧪 Testes:")
print(f"Slot 1  → {slot_to_day_and_block(1)}  (Segunda, 09h-11h)")
print(f"Slot 5  → {slot_to_day_and_block(5)}  (Terça, 09h-11h)")
print(f"Slot 20 → {slot_to_day_and_block(20)} (Sexta, 16h-18h)")

## Leitura do Dataset

In [ ]:
def load_dataset(path: str) -> Dict:
    """
    Lê o ficheiro e constrói mapas base e derivados:
      - class_to_ucs: turma -> [UCs]
      - teacher_to_ucs: docente -> [UCs]
      - teacher_unavail: docente -> set(slots indisponíveis)
      - uc_room_required: UC -> sala obrigatória
      - uc_online_idx: UC -> set({1,2}) índices de aulas online
      - uc_to_class / uc_to_teacher
      - UCs: lista de todas as UCs (ordenada)
    """
    raw = pathlib.Path(path).read_text(encoding="utf-8")

    cc   = read_section(raw, "cc")
    dsd  = read_section(raw, "dsd")
    tr   = read_section(raw, "tr")
    rr   = read_section(raw, "rr")
    oc   = read_section(raw, "oc")

    class_to_ucs = {}
    for ln in cc:
        parts = ln.split()
        class_to_ucs[parts[0]] = parts[1:]

    teacher_to_ucs = {}
    for ln in dsd:
        parts = ln.split()
        teacher_to_ucs[parts[0]] = parts[1:]

    teacher_unavail = {}
    for ln in tr:
        parts = ln.split()
        teacher_unavail[parts[0]] = set(map(int, parts[1:]))

    uc_room_required = {}
    for ln in rr:
        uc, room = ln.split()
        uc_room_required[uc] = room

    uc_online_idx = defaultdict(set)
    for ln in oc:
        uc, idx = ln.split()
        uc_online_idx[uc].add(int(idx))

    uc_to_class = {}
    for c, ucs in class_to_ucs.items():
        for uc in ucs:
            uc_to_class[uc] = c

    uc_to_teacher = {}
    for t, ucs in teacher_to_ucs.items():
        for uc in ucs:
            uc_to_teacher[uc] = t

    UCs = sorted(uc_to_class.keys())

    return {
        "class_to_ucs": class_to_ucs,
        "teacher_to_ucs": teacher_to_ucs,
        "teacher_unavail": teacher_unavail,
        "uc_room_required": uc_room_required,
        "uc_online_idx": uc_online_idx,
        "uc_to_class": uc_to_class,
        "uc_to_teacher": uc_to_teacher,
        "UCs": UCs
    }

### Carregar Dados

In [ ]:
try:
    data = load_dataset(DATA_PATH)
    print("Dataset carregado com sucesso!\n")
    print(f"Turmas: {list(data['class_to_ucs'].keys())}")
    print(f"Docentes: {list(data['teacher_to_ucs'].keys())}")
    print(f"Total UCs: {len(data['UCs'])}")
    print(f"UCs com sala obrigatória: {list(data['uc_room_required'].keys())}")
    print(f"UCs com aulas online: {list(data['uc_online_idx'].keys())}")
except FileNotFoundError:
    print(f"Erro: não encontrei '{DATA_PATH}'. Coloca o ficheiro na mesma pasta.")

## Validação e Diagnóstico

In [ ]:
def sanity_check_data(data: Dict) -> bool:
    """Verifica: UC tem turma/docente; slots de indisponibilidade estão em 1..20."""
    ok = True
    UCs = data["UCs"]
    uc_to_class   = data["uc_to_class"]
    uc_to_teacher = data["uc_to_teacher"]
    teacher_unav  = data["teacher_unavail"]

    for uc in UCs:
        if uc not in uc_to_class:
            print(f"[SANITY] UC sem turma: {uc}"); ok = False
        if uc not in uc_to_teacher:
            print(f"[SANITY] UC sem docente: {uc}"); ok = False

    for t, ss in teacher_unav.items():
        for s in ss:
            if s not in SLOTS:
                print(f"[SANITY] Slot {s} fora do intervalo 1..{len(SLOTS)} (docente {t})")
                ok = False

    for uc, room in data["uc_room_required"].items():
        if not room or not isinstance(room, str):
            print(f"[SANITY] Sala obrigatória inválida em {uc}: {room!r}")
            ok = False

    return ok

# Executar validação
if sanity_check_data(data):
    print("Validação: Todos os dados estão consistentes!")
else:
    print("Validação: Erros encontrados no dataset!")

In [ ]:
def compute_var_infos(data: Dict, base_rooms=("SalaA","SalaB"), split_week=False) -> List[Dict]:
    """
    Para cada UC_i (i=1,2) calcula:
      - modo (online/presencial)
      - slots válidos (após indisponibilidades)
      - salas possíveis
      - tamanho do domínio
    """
    uc_to_class    = data["uc_to_class"]
    uc_to_teacher  = data["uc_to_teacher"]
    uc_room_req    = data["uc_room_required"]
    teacher_unav   = data["teacher_unavail"]
    uc_online_idx  = data["uc_online_idx"]
    UCs            = data["UCs"]

    var_infos = []
    for uc in UCs:
        for i in (1, 2):
            name = f"{uc}_{i}"
            teacher = uc_to_teacher[uc]
            turma = uc_to_class[uc]
            bad = teacher_unav.get(teacher, set())
            valid_slots = [s for s in SLOTS if s not in bad]

            if split_week:
                mid = len(SLOTS) // 2
                pivot = SLOTS[mid-1]
                if i == 1:
                    valid_slots = [s for s in valid_slots if s <= pivot]
                else:
                    valid_slots = [s for s in valid_slots if s > pivot]

            is_online = i in uc_online_idx.get(uc, set())
            mode = "online" if is_online else "presencial"

            if is_online:
                rooms = [f"Online::{uc}"]
            else:
                rooms = [uc_room_req[uc]] if uc in uc_room_req else list(base_rooms)

            domain = [(s, r, mode) for s in sorted(valid_slots) for r in sorted(rooms)]
            var_infos.append({
                "name": name,
                "mode": mode,
                "teacher": teacher,
                "turma": turma,
                "valid_slots": sorted(valid_slots),
                "rooms": rooms,
                "domain_size": len(domain),
                "sample": domain[:min(5, len(domain))]
            })
    return var_infos

# Executar diagnóstico
var_infos = compute_var_infos(data)
print("\nDiagnóstico das Variáveis:")
for vi in var_infos[:5]:  # Mostrar apenas as primeiras 5
    print(f"  {vi['name']:10} | {vi['mode']:10} | Docente: {vi['teacher']:6} | Domínio: {vi['domain_size']} opções")

## Resolução do CSP

Esta é a parte mais importante! Vamos criar o problema CSP e adicionar as restrições.

In [ ]:
def build_csp(data: Dict, soft_max3=True, ignore_rooms=False):
    """
    Constrói o problema CSP com todas as restrições:
    - Docente não pode estar em 2 sítios ao mesmo tempo
    - Turma não pode ter 2 aulas ao mesmo tempo
    - Aulas da mesma UC em dias diferentes
    - Máximo 3 blocos por dia (soft)
    """
    prob = Problem()
    var_infos = compute_var_infos(data, split_week=False)
    
    # Adicionar variáveis com domínios
    for vi in var_infos:
        domain = [(s, r, vi["mode"]) for s in vi["valid_slots"] for r in vi["rooms"]]
        if ignore_rooms:
            domain = [(s, "_", vi["mode"]) for s in vi["valid_slots"]]
        prob.addVariable(vi["name"], domain)
    
    # Agrupar variáveis por docente e turma
    by_teacher = defaultdict(list)
    by_class = defaultdict(list)
    for vi in var_infos:
        by_teacher[vi["teacher"]].append(vi["name"])
        by_class[vi["turma"]].append(vi["name"])
    
    # RESTRIÇÃO 1: Docente não pode ter 2 aulas no mesmo slot
    for teacher, tvars in by_teacher.items():
        for i, v1 in enumerate(tvars):
            for v2 in tvars[i+1:]:
                prob.addConstraint(lambda a, b: a[0] != b[0], (v1, v2))
    
    # RESTRIÇÃO 2: Turma não pode ter 2 aulas no mesmo slot
    for turma, cvars in by_class.items():
        for i, v1 in enumerate(cvars):
            for v2 in cvars[i+1:]:
                prob.addConstraint(lambda a, b: a[0] != b[0], (v1, v2))
    
    # RESTRIÇÃO 3: Aulas da mesma UC em dias diferentes
    for uc in data["UCs"]:
        v1, v2 = f"{uc}_1", f"{uc}_2"
        prob.addConstraint(lambda a, b: slot_day(a[0]) != slot_day(b[0]), (v1, v2))
    
    # RESTRIÇÃO 4 (SOFT): Máximo 3 blocos por dia por turma
    if soft_max3:
        for turma, cvars in by_class.items():
            def max3_constraint(*vals):
                day_count = defaultdict(int)
                for v in vals:
                    day_count[slot_day(v[0])] += 1
                return all(c <= 3 for c in day_count.values())
            prob.addConstraint(max3_constraint, cvars)
    
    return prob, by_class

print("Função build_csp() definida!")

### Resolver o CSP

In [ ]:
print("A procurar solução...")
start = time.time()

# Construir e resolver
prob, by_class = build_csp(data, soft_max3=True, ignore_rooms=False)
sol = prob.getSolution()

elapsed = time.time() - start

if sol:
    print(f"Solução encontrada em {elapsed:.2f}s!")
    print(f"Total de variáveis atribuídas: {len(sol)}")
else:
    print(f"Nenhuma solução encontrada em {elapsed:.2f}s")

## Visualização da Solução

In [ ]:
def show_by_class(sol: Dict, by_class: Dict):
    """Mostra horário organizado por turma."""
    print("\n" + "="*60)
    print("HORÁRIOS POR TURMA")
    print("="*60)
    
    for turma in sorted(by_class.keys()):
        print(f"\nTurma: {turma}")
        print("-" * 60)
        
        schedule = defaultdict(list)
        for var in by_class[turma]:
            uc = var.split("_")[0]
            slot, room, mode = sol[var]
            day, block = slot_to_day_and_block(slot)
            time = BLOCK_TO_TIME[block]
            schedule[day].append((block, uc, room, mode, time))
        
        for day in DAYS:
            if day in schedule:
                print(f"\n  {day}:")
                for block, uc, room, mode, time in sorted(schedule[day]):
                    marker = "💻" if mode == "online" else "🏫"
                    print(f"    {marker} {block} ({time}): {uc:6} @ {room}")

if sol:
    show_by_class(sol, by_class)

In [ ]:
def show_by_teacher(sol: Dict, data: Dict):
    """Mostra horário organizado por docente."""
    print("\n" + "="*60)
    print("HORÁRIOS POR DOCENTE")
    print("="*60)
    
    by_teacher = defaultdict(list)
    for var, (slot, room, mode) in sol.items():
        uc = var.split("_")[0]
        teacher = data["uc_to_teacher"][uc]
        by_teacher[teacher].append((slot, uc, room, mode, var))
    
    for teacher in sorted(by_teacher.keys()):
        print(f"\nDocente: {teacher}")
        print("-" * 60)
        
        schedule = defaultdict(list)
        for slot, uc, room, mode, var in by_teacher[teacher]:
            day, block = slot_to_day_and_block(slot)
            time = BLOCK_TO_TIME[block]
            schedule[day].append((block, uc, room, mode, time))
        
        for day in DAYS:
            if day in schedule:
                print(f"\n  {day}:")
                for block, uc, room, mode, time in sorted(schedule[day]):
                    marker = "💻" if mode == "online" else "🏫"
                    print(f"    {marker} {block} ({time}): {uc:6} @ {room}")

if sol:
    show_by_teacher(sol, data)

## Exportação para CSV, PNG e PDF

In [ ]:
def build_calendar_frames(sol: Dict, by_class: Dict, data: Dict) -> Dict[str, pd.DataFrame]:
    """Cria DataFrame por turma para visualização em tabela."""
    frames = {}
    for turma, tvars in by_class.items():
        grid = {blk: {day: "" for day in DAYS} for blk in BLOCK_LABELS}
        for var in tvars:
            uc = var.split("_")[0]
            slot, room, mode = sol[var]
            day, blk = slot_to_day_and_block(slot)
            label = f"{uc} @{'ONLINE' if mode=='online' else room}{' (online)' if mode=='online' else ''}"
            grid[blk][day] = (grid[blk][day] + " | " if grid[blk][day] else "") + label
        df = pd.DataFrame({blk: [grid[blk][d] for d in DAYS] for blk in BLOCK_LABELS}, index=DAYS)
        frames[turma] = df
    return frames

# Cores por UC
def uc_to_rgb(uc: str) -> Tuple[float,float,float]:
    """Converte o nome da UC numa cor HSV → RGB estável."""
    h = int(hashlib.md5(uc.encode("utf-8")).hexdigest(), 16)
    hue = (h % 360) / 360.0
    sat, val = 0.45, 0.95
    return colorsys.hsv_to_rgb(hue, sat, val)

def luminance(rgb: Tuple[float,float,float]) -> float:
    """Luminância para decidir cor do texto."""
    r, g, b = rgb
    return 0.2126*r + 0.7152*g + 0.0722*b

def text_color_for_bg(rgb: Tuple[float,float,float]) -> str:
    """Preto em fundos claros; branco em fundos escuros."""
    return "black" if luminance(rgb) > 0.6 else "white"

if sol:
    frames = build_calendar_frames(sol, by_class, data)
    print("\nDataFrames criados para cada turma:")
    for turma in frames.keys():
        print(f"  ✓ {turma}")

### Visualizar Horário de uma Turma

In [ ]:
if sol:
    # Mostrar horário da primeira turma
    turma_exemplo = list(frames.keys())[0]
    print(f"\nHorário da turma {turma_exemplo}:\n")
    display(frames[turma_exemplo])

In [ ]:
def _render_df_as_figure_colored(df: pd.DataFrame, title: str, figsize=(12, 6)):
    """Renderiza tabela colorida com matplotlib."""
    fig, ax = plt.subplots(figsize=figsize, dpi=120)
    ax.axis('off')
    
    cell_text = df.values
    row_labels = [f"{d}" for d in df.index]
    col_labels = [f"{c}\n{BLOCK_TO_TIME[c]}" for c in df.columns]
    
    tbl = ax.table(cellText=cell_text,
                   rowLabels=row_labels,
                   colLabels=col_labels,
                   loc='center',
                   cellLoc='center')
    
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(9)
    tbl.scale(1.1, 2.0)
    
    # Cabeçalhos
    for (i, j), cell in tbl.get_celld().items():
        if i == 0 or j == -1:
            cell.set_text_props(fontweight='bold')
            cell.set_facecolor((0.92, 0.92, 0.95))
    
    # Colorir células por UC
    n_rows, n_cols = df.shape
    for ridx in range(n_rows):
        for cidx in range(n_cols):
            cell = tbl[ridx+1, cidx]
            txt = df.iat[ridx, cidx]
            if not txt:
                cell.set_facecolor((1, 1, 1))
                continue
            
            parts = [p.strip() for p in txt.split("|")]
            if len(parts) == 1:
                uc = parts[0].split()[0]
                bg = uc_to_rgb(uc)
                cell.set_facecolor(bg)
                cell.get_text().set_color(text_color_for_bg(bg))
                if "@ONLINE" in parts[0]:
                    cell.set_hatch("///")
            else:
                cell.set_facecolor((0.85, 0.85, 0.85))
    
    plt.title(title, fontsize=14, fontweight='bold', pad=20)
    plt.tight_layout()
    return fig

# Visualizar horário colorido
if sol:
    fig = _render_df_as_figure_colored(frames[turma_exemplo], f"Horário {turma_exemplo}")
    plt.show()

### Exportar Todos os Horários

In [ ]:
def export_all(sol, by_class, data, outdir="export"):
    """Exporta horários em CSV, PNG e PDF."""
    frames = build_calendar_frames(sol, by_class, data)
    
    # CSV
    csv_dir = os.path.join(outdir, "csv")
    os.makedirs(csv_dir, exist_ok=True)
    for turma, df in frames.items():
        df.to_csv(os.path.join(csv_dir, f"horario_{turma}.csv"), encoding="utf-8")
    
    # PNG
    img_dir = os.path.join(outdir, "img")
    os.makedirs(img_dir, exist_ok=True)
    for turma, df in frames.items():
        fig = _render_df_as_figure_colored(df, f"Horário {turma}")
        fig.savefig(os.path.join(img_dir, f"horario_{turma}.png"), bbox_inches="tight", dpi=150)
        plt.close(fig)
    
    # PDF
    pdf_dir = os.path.join(outdir, "pdf")
    os.makedirs(pdf_dir, exist_ok=True)
    for turma, df in frames.items():
        fig = _render_df_as_figure_colored(df, f"Horário {turma}")
        fig.savefig(os.path.join(pdf_dir, f"horario_{turma}.pdf"), bbox_inches="tight")
        plt.close(fig)
    
    print(f"\nExportação completa!")
    print(f"CSV → {csv_dir}")
    print(f"PNG → {img_dir}")
    print(f"PDF → {pdf_dir}")

if sol:
    export_all(sol, by_class, data, outdir="export")

## Análise da Solução

In [ ]:
if sol:
    print("\nESTATÍSTICAS DA SOLUÇÃO\n")
    print(f"Total de aulas agendadas: {len(sol)}")
    
    # Distribuição por dia
    day_count = defaultdict(int)
    for var, (slot, room, mode) in sol.items():
        day = slot_day(slot)
        day_count[day] += 1
    
    print("\nAulas por dia:")
    for day in DAYS:
        print(f"  {day}: {'█' * day_count[day]} ({day_count[day]})")
    
    # Aulas online vs presenciais
    online_count = sum(1 for _, (_, _, mode) in sol.items() if mode == "online")
    presencial_count = len(sol) - online_count
    
    print(f"\nAulas online: {online_count}")
    print(f"Aulas presenciais: {presencial_count}")